# Lab 1 — Evaluate a black-box retriever

**Companion to course Lessons 1 and 2.**

> *Imagine you're building a recruiting application. A hiring
> manager needs a shortlist by the end of the day. You run your
> retrieval system and get back a ranked list of candidates.
> But how do you know if it's any good?*

That's the question this lab answers, end-to-end, on real
data. The "retrieval system" is **lexical (BM25) search** —
the keyword-matching family of algorithms that has powered
full-text search for decades. We treat it as a black box: text
in, ranked list of documents out. Our job is to **score** that
ranked list against ground-truth relevance judgements.

You'll compute **Precision@k**, **Recall@k**, **NDCG@k**, and
**MRR** by hand on one query (so you can see exactly how the
math works), then aggregate across many queries.

## Step 1 — Setup

Match `DATASET` to whatever you ingested in Lab 0.

In [ ]:
import os, sys
# Notebooks live at the repo root; library modules live in src/.
_REPO_ROOT = os.path.abspath(os.getcwd())
_SRC = os.path.join(_REPO_ROOT, "src")
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)

In [ ]:
from dotenv import load_dotenv
load_dotenv(os.path.join(_REPO_ROOT, ".env"))

import pymongo
from lib import load_beir_dataset

DB_NAME           = "voyage_context_demo"
DATASET           = "scifact"
COLL_NAME         = f"chunks_{DATASET.replace('-', '_')}"
TEXT_INDEX_NAME   = "voyage_text_index"

client = pymongo.MongoClient(os.environ["MONGODB_URI"])
coll   = client[DB_NAME][COLL_NAME]
corpus, queries, qrels, info = load_beir_dataset(DATASET)

print(f"Connected to {DB_NAME}.{COLL_NAME} — {coll.estimated_document_count():,} chunks")
print(f"Loaded {DATASET}: {len(queries)} test queries, qrels for {len(qrels)} of them")

## Step 2 — What is an evaluation dataset?

Every BEIR dataset is three Python dicts:

- **`corpus`** — the haystack: `{doc_id: {"title": ..., "text": ...}}`.
- **`queries`** — what users ask: `{query_id: query_text}`.
- **`qrels`** ("query relevance" judgements) — the answer key:
  `{query_id: {doc_id: relevance_score}}`. A non-zero score
  means a human judged that document relevant to that query.

The qrels are sometimes called the **judgement list**, **golden
dataset**, or **ground truth**. Without them, evaluation isn't
possible — there's nothing to compare retrieval results against.

Pull one query and look at it:

In [ ]:
# We want a worked-example query where BM25 actually retrieves at
# least one relevant doc in the top 10 — otherwise the metric demo
# below would be all zeros. (Some queries share no keywords with
# their relevant docs; BM25 can't bridge that gap. Those are great
# examples of where vector search wins, which we'll see in Lab 2.)
def has_top_10_hit(qid):
    q_qrels = qrels.get(qid, {})
    rel_set = {did for did, s in q_qrels.items() if s > 0}
    if not rel_set:
        return False
    pipeline = [
        {"$search": {"index": TEXT_INDEX_NAME,
                      "text": {"path": "text", "query": queries[qid]}}},
        {"$limit": 40},
    ]
    seen_docs = set()
    for row in coll.aggregate(pipeline):
        seen_docs.add(row['doc_id'])
        if len(seen_docs) >= 10:
            break
    return bool(seen_docs & rel_set)

sample_qid    = next(qid for qid in queries if has_top_10_hit(qid))
sample_query  = queries[sample_qid]
sample_qrels  = qrels.get(sample_qid, {})
relevant_docs = {did: s for did, s in sample_qrels.items() if s > 0}

print(f"Query ID     : {sample_qid}")
print(f"Query text   : {sample_query!r}")
print(f"Relevant docs: {len(relevant_docs)}")
for did, score in list(relevant_docs.items())[:3]:
    title = corpus.get(did, {}).get('title', '<missing>')
    print(f"  {did:<12} grade={score}  title={title[:60]!r}")

### Graded vs binary relevance

BEIR qrels use **graded relevance**:

- `0` — not relevant
- `1` — relevant
- `2` — highly relevant (in datasets that distinguish)

Binary relevance treats anything `> 0` as relevant. Most
metrics work with either form; NDCG benefits the most from
the extra resolution. We'll see why below.

## Step 3 — Run the BM25 black box

Atlas Search's `$search` stage runs a BM25-family query
against the text index we built in Lab 0. The aggregation
pipeline is the entire interface — no library functions —
and looks like this:

In [ ]:
TOP_K = 10

pipeline = [
    # Stage 1: lexical (BM25) search against our text index.
    {"$search": {
        "index": TEXT_INDEX_NAME,
        "text" : {"path": "text", "query": sample_query},
    }},
    # Stage 2: pull the BM25 score from the index metadata into a regular field.
    {"$addFields": {"score": {"$meta": "searchScore"}}},
    # Stage 3: keep the top N candidates. We over-fetch and dedupe below.
    {"$limit": TOP_K * 4},
    {"$sort":  {"score": -1}},
]

results = list(coll.aggregate(pipeline))
print(f"$search returned {len(results)} chunks (before dedup).")

Our collection stores **chunks**, not parent documents — one
BEIR doc may have produced several chunks. So we dedupe to the
best chunk per parent doc, then keep the top-K.

In [ ]:
# Keep the highest-scoring chunk per parent doc.
seen = {}
for row in results:
    did = row['doc_id']
    if did not in seen or row['score'] > seen[did]['score']:
        seen[did] = row

ranked = sorted(seen.values(), key=lambda r: r['score'], reverse=True)[:TOP_K]

print(f"Top-{TOP_K} for query {sample_qid!r}:")
for rank, row in enumerate(ranked, 1):
    grade = sample_qrels.get(row['doc_id'], 0)
    tag = "★" if grade > 0 else " "
    print(f"  {rank:>2}. {tag} doc {row['doc_id']:<12} "
          f"score={row['score']:>6.3f}  grade={grade}  "
          f"{row['title'][:55]!r}")

Rows marked ★ are ones the qrels confirm are relevant.
Unmarked rows are either confirmed irrelevant (grade 0) or
**not judged at all** — most public IR datasets only have
judgements for a small subset of the corpus. That's the
pooling assumption: docs that no retriever surfaced are
treated as irrelevant.

## Step 4 — Precision@k

> *Of the documents the system returned, what fraction are relevant?*

Precision answers the **false positive** question: "how much
noise is in my shortlist?" If 4 of 5 returned candidates are
relevant, Precision@5 = 0.8.

$$\text{Precision@k} = \frac{\#\text{ relevant in top-}k}{k}$$

In [ ]:
ranked_ids   = [r['doc_id'] for r in ranked]
relevant_ids = {did for did, score in sample_qrels.items() if score > 0}

k = 5
top_k = ranked_ids[:k]
hits  = sum(1 for did in top_k if did in relevant_ids)
precision_at_5 = hits / k

print(f"top-5 doc_ids : {top_k}")
print(f"# relevant    : {hits}")
print(f"Precision@5   = {hits}/{k} = {precision_at_5:.3f}")

## Step 5 — Recall@k

> *Of all the relevant documents that exist, what fraction did we return?*

Recall answers the **false negative** question: "how many
relevant documents did I miss?" If 10 docs in the corpus are
relevant and 7 appear in the top-k, Recall@k = 0.7.

$$\text{Recall@k} = \frac{\#\text{ relevant in top-}k}{\#\text{ relevant in corpus}}$$

> **Note.** "Recall" also shows up in vector-index benchmarks
> where it means the fraction of true nearest neighbours an
> approximate index returned. Same word, different reference
> point — that's an index-quality metric; this is a
> retrieval-quality metric.

In [ ]:
k = 10
top_k         = ranked_ids[:k]
hits_in_top_k = len(set(top_k) & relevant_ids)
recall_at_10  = hits_in_top_k / len(relevant_ids)

print(f"# relevant in corpus : {len(relevant_ids)}")
print(f"# relevant in top-{k}  : {hits_in_top_k}")
print(f"Recall@{k}            = {hits_in_top_k}/{len(relevant_ids)} = {recall_at_10:.3f}")

### Precision/Recall trade-off

These two pull in opposite directions. Want higher precision?
Return fewer, higher-confidence results — recall drops. Want
higher recall? Cast a wider net — precision drops. Which
matters more is an application-level decision: a legal-discovery
tool can't miss relevant precedent (favour recall); a
user-facing search box can't bury good results under junk
(favour precision).

## Step 6 — NDCG@k (the most important one)

Precision and Recall treat the top-k as a *set* — they ignore
order. But users look at ranked lists top-down. A relevant
document at rank 1 is much more valuable than the same
document at rank 9.

**NDCG** captures that. It has three pieces:

1. **Gain** — the relevance grade of each retrieved document
   (graded relevance pays off here).
2. **Discount** — divide the gain by `log₂(rank + 1)`, so
   rank 1 weighs 1.0, rank 5 weighs ~0.39, rank 10 weighs ~0.29.
3. **Normalize** — divide by the score of an *ideal* ranking
   (relevant docs sorted from highest grade to lowest), so the
   final score sits in `[0, 1]`. A perfect ranking scores 1.0.

$$\text{DCG@k} = \sum_{i=1}^{k} \frac{\text{grade}(d_i)}{\log_2(i+1)}
\qquad \text{NDCG@k} = \frac{\text{DCG@k}}{\text{IDCG@k}}$$

In [ ]:
import math

k = 5
print(f"DCG@{k} calculation:")
dcg = 0.0
for rank, did in enumerate(ranked_ids[:k], 1):
    grade    = sample_qrels.get(did, 0)
    discount = math.log2(rank + 1)
    term     = grade / discount
    dcg     += term
    print(f"  rank {rank}: grade={grade}  log2({rank+1})={discount:.3f}  "
          f"term={grade}/{discount:.3f} = {term:.3f}")
print(f"DCG@{k}  = {dcg:.3f}")

# The ideal ranking: every relevant doc sorted by descending grade.
ideal_grades = sorted((s for s in sample_qrels.values() if s > 0), reverse=True)[:k]
idcg = sum(g / math.log2(i + 1) for i, g in enumerate(ideal_grades, 1))
print(f"IDCG@{k} = {idcg:.3f}   (best possible ordering of available grades)")
print(f"NDCG@{k} = DCG/IDCG = {dcg/idcg if idcg else 0:.3f}")

NDCG@k is *the* primary metric for embedding-model comparison.
You'll see it on every public leaderboard (MTEB, RTEB, BEIR
itself). It rewards putting the most relevant documents at the
top while still tolerating burying weaker (but still-relevant)
ones lower in the list.

## Step 7 — MRR (Mean Reciprocal Rank)

> *On average, how high up does the first relevant document appear?*

Scan the ranked list from position 1 downward until you hit a
relevant doc. The reciprocal rank is `1 / first_rel_rank`:
1.0 at position 1, 0.5 at position 2, 0.33 at position 3.
Average across all queries to get MRR.

MRR is most informative when there's one clearly best fit per
query — a lookup, a known-item search, or when evaluating a
reranker whose only job is to bubble the best answer to the
top.

In [ ]:
first_rel_rank = next(
    (i for i, did in enumerate(ranked_ids, 1) if did in relevant_ids),
    None,
)
if first_rel_rank:
    rr = 1.0 / first_rel_rank
    print(f"First relevant doc at rank {first_rel_rank}  →  RR = 1/{first_rel_rank} = {rr:.3f}")
else:
    rr = 0.0
    print("No relevant doc retrieved at all  →  RR = 0")

## Step 8 — Evaluate over many queries

A retrieval system isn't judged on one query — it's judged on
a distribution. We'll wrap the per-query math into a single
function, then loop over the first `N` test queries.

This is the function you'll re-use in Labs 2 and 3.

In [ ]:
import math

def query_metrics(ranked: list[str], qrels: dict[str, int], ks=(5, 10)) -> dict:
    """Compute P@k, R@k, NDCG@k for each k, plus MRR and AP."""
    rel_set = {did for did, s in qrels.items() if s > 0}
    out = {}
    for k in ks:
        top_k = ranked[:k]
        out[f"P@{k}"] = sum(1 for d in top_k if d in rel_set) / k
        out[f"R@{k}"] = len(set(top_k) & rel_set) / max(1, len(rel_set))
        # NDCG@k
        dcg = sum(qrels.get(d, 0) / math.log2(i + 1)
                  for i, d in enumerate(top_k, 1))
        ideal = sorted((s for s in qrels.values() if s > 0), reverse=True)[:k]
        idcg = sum(g / math.log2(i + 1) for i, g in enumerate(ideal, 1))
        out[f"NDCG@{k}"] = dcg / idcg if idcg else 0.0
    # MRR: 1 / rank of first relevant
    out["MRR"] = next(
        (1.0 / i for i, d in enumerate(ranked, 1) if d in rel_set), 0.0,
    )
    # AP: mean of precision-at-each-hit
    cum = hits = 0
    for i, d in enumerate(ranked, 1):
        if d in rel_set:
            hits += 1
            cum  += hits / i
    out["AP"] = cum / len(rel_set) if rel_set else 0.0
    return out

# Sanity check on the query we already analysed.
print(query_metrics(ranked_ids, sample_qrels))

In [ ]:
def bm25_search(query_text: str, top_k: int = 10) -> list[str]:
    """Run a $search pipeline and return deduplicated doc_ids."""
    pipeline = [
        {"$search": {
            "index": TEXT_INDEX_NAME,
            "text" : {"path": "text", "query": query_text},
        }},
        {"$addFields": {"score": {"$meta": "searchScore"}}},
        {"$limit": top_k * 4},
        {"$sort":  {"score": -1}},
    ]
    seen = {}
    for row in coll.aggregate(pipeline):
        did = row['doc_id']
        if did not in seen or row['score'] > seen[did]['score']:
            seen[did] = row
    return [r['doc_id'] for r in sorted(seen.values(),
            key=lambda r: r['score'], reverse=True)[:top_k]]


N_QUERIES = 30
query_ids = list(queries.keys())[:N_QUERIES]
per_query = []
for qid in query_ids:
    ranked = bm25_search(queries[qid], top_k=10)
    per_query.append(query_metrics(ranked, qrels.get(qid, {})))

# Aggregate: mean each metric; AP-mean becomes MAP.
keys = per_query[0].keys()
agg = {k: sum(q[k] for q in per_query) / len(per_query) for k in keys}
agg["MAP"] = agg.pop("AP")
print(f"BM25 lexical retrieval — {DATASET}, {N_QUERIES} queries")
for k, v in agg.items():
    print(f"  {k:<8} {v:.3f}")

## Reading the numbers

Each metric tells you something different:

- **P@5 / P@10** — top-of-list cleanliness (low = lots of false positives).
- **R@5 / R@10** — coverage of the relevant set (low = lots of false negatives).
- **NDCG@5 / NDCG@10** — ranking quality. The number you'd
  quote when comparing two retrievers on a benchmark.
- **MRR** — how prominently the first relevant result sits.
- **MAP** — Mean Average Precision; recall-and-rank summary
  across all positions where relevant docs land.

Write these numbers down — you'll compare them to **vector**
and **hybrid** retrievers in Lab 2.

**Next:** open **`02_swap_blackbox.ipynb`**.